<a href="https://colab.research.google.com/github/GOWTHAMVANTAKULA/Finance_Sentiment_Analyzer/blob/main/Finance_Sentiment_Analyzer_TFIDF_%2B_ML_RNN_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **DATA**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/FINANCE_NLP_DL/all-data.csv', encoding='latin-1',
                   header = None,names = ['label','text'])

In [ ]:
data.head()

,label,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


In [ ]:
data['label'].value_counts()

,count
label,
neutral,2879
positive,1363
negative,604


In [ ]:
data['text'] = data['text'].str.lower()

In [ ]:
label_map = {'negative' : 0, 'neutral' : 1, 'positive' : 2}
data['label'] = data['label'].map(label_map)

In [ ]:
X = data['text']
y = data['label']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)


### **TF-IDF & LOGESTIC REGRESSION**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()

In [ ]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

In [ ]:
model.fit(X_train_vec,y_train)

LogisticRegression()

In [ ]:
y_pred = model.predict(X_test_vec)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.90      0.48      0.63       110
           1       0.75      0.95      0.84       571
           2       0.80      0.52      0.63       289

    accuracy                           0.77       970
   macro avg       0.82      0.65      0.70       970
weighted avg       0.78      0.77      0.75       970



### **RNN**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



**STEP - 1 == TOKENIZATION**

In [ ]:
# initialize tokenizer
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(X_train)



In [ ]:
# CONVERT TEXT TO SEQUENCE
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)



**PADDING - MAKE ALL SEQUENCES SAME LENGTH**

In [ ]:
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen = max_len)
X_test_pad = pad_sequences(X_test_seq,maxlen = max_len)

**ONE - HOT ENCODING TO THE TARGET VARABLES**

In [ ]:
from tensorflow.keras.utils import to_categorical
y_train_ohe = to_categorical(y_train, num_classes = 3)
y_test_ohe = to_categorical(y_test, num_classes = 3)

**BUILD RNN MODEL**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense




In [ ]:
model = Sequential()
model.add(Embedding(input_dim = 5000,output_dim = 64, input_length = max_len))
model.add(SimpleRNN(64))
model.add(Dense(3,activation = 'softmax'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
model.fit(X_train_pad,y_train_ohe,epochs = 5,batch_size = 32,validation_split = 0.2)



Epoch 1/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.6010 - loss: 0.8964 - val_accuracy: 0.6430 - val_loss: 0.8508
Epoch 2/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7755 - loss: 0.5633 - val_accuracy: 0.5966 - val_loss: 0.8850
Epoch 3/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9568 - loss: 0.1674 - val_accuracy: 0.6198 - val_loss: 1.0321
Epoch 4/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9903 - loss: 0.0518 - val_accuracy: 0.6198 - val_loss: 1.1278
Epoch 5/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9945 - loss: 0.0335 - val_accuracy: 0.6147 - val_loss: 1.1921


In [ ]:
# prediction of X_test
y_pred = model.predict(X_test_pad)


31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step


In [ ]:
# convert to lables
import numpy as np
y_pred_lables = np.argmax(y_pred,axis = 1)


In [ ]:
# convert y_test_ohe to lables
y_test_labels = np.argmax(y_test_ohe, axis=1)



In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test_labels,y_pred_lables))

              precision    recall  f1-score   support

           0       0.47      0.40      0.43       110
           1       0.73      0.83      0.78       571
           2       0.46      0.37      0.41       289

    accuracy                           0.64       970
   macro avg       0.56      0.53      0.54       970
weighted avg       0.62      0.64      0.63       970



### **LSTM MODEL**

In [ ]:
from tensorflow.keras.layers import LSTM

model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=64, input_length=max_len))
model.add(LSTM(64))
model.add(Dense(3, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train_pad, y_train_ohe, epochs=5, batch_size=32, validation_split=0.2)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


97/97 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.6145 - loss: 0.8810 - val_accuracy: 0.6598 - val_loss: 0.7965
Epoch 2/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - accuracy: 0.7213 - loss: 0.6351 - val_accuracy: 0.6508 - val_loss: 0.7759
Epoch 3/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.8506 - loss: 0.3921 - val_accuracy: 0.6985 - val_loss: 0.7439
Epoch 4/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 9s 98ms/step - accuracy: 0.9206 - loss: 0.2321 - val_accuracy: 0.7165 - val_loss: 0.7800
Epoch 5/5
97/97 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.9597 - loss: 0.1318 - val_accuracy: 0.7178 - val_loss: 0.9924


In [ ]:
y_pred = model.predict(X_test_pad)



31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


In [ ]:
import numpy as np
y_pred_lables = np.argmax(y_pred,axis = 1)

In [ ]:
y_test_labels = np.argmax(y_test_ohe, axis=1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test_labels,y_pred_lables))

              precision    recall  f1-score   support

           0       0.71      0.61      0.65       110
           1       0.77      0.88      0.82       571
           2       0.69      0.54      0.61       289

    accuracy                           0.75       970
   macro avg       0.72      0.67      0.69       970
weighted avg       0.74      0.75      0.74       970

